# 05 – Advanced Models
Sentence-BERT model, XGBoost hybrid and DistilBERT. Trained on train, compared on validation; test is not used here.

## Setup

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import pandas as pd, numpy as np, os, time, joblib
from datetime import datetime
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import f1_score, accuracy_score, classification_report
from xgboost import XGBClassifier

base = "/content/drive/MyDrive/resume-jd-matching"
data_dir, model_dir = f"{base}/data", f"{base}/models"
log_path = f"{base}/reports/experiments.csv"

train = pd.read_parquet(f"{data_dir}/train_feat.parquet")
val = pd.read_parquet(f"{data_dir}/val_feat.parquet")
order = ["No Fit", "Potential Fit", "Good Fit"]
y_tr, y_va = train["label"], val["label"]

FEATS = ["skill_coverage", "skill_jaccard", "matched_count", "missing_count", "jd_skill_count",
         "resume_skill_count", "jd_years", "resume_years", "exp_gap", "exp_meets",
         "jd_years_missing", "resume_years_missing", "tfidf_cosine", "embed_cosine"]

def report(name, pred, features, params="", fit_s=0.0, verbose=True):
    f1, acc = f1_score(y_va, pred, average="macro"), accuracy_score(y_va, pred)
    row = {"date": datetime.now().strftime("%Y-%m-%d %H:%M"), "notebook": "05", "model": name,
           "features": features, "params": params, "val_macro_f1": round(f1, 4),
           "val_accuracy": round(acc, 4), "fit_seconds": round(fit_s, 1)}
    pd.DataFrame([row]).to_csv(log_path, mode="a", header=not os.path.exists(log_path), index=False)
    print(f"{name}: macro F1 {f1:.3f} | accuracy {acc:.3f}")
    if verbose:
        print(classification_report(y_va, pred, labels=order, digits=3, zero_division=0))
    return f1

print(train.shape, val.shape)

Mounted at /content/drive
(5285, 19) (943, 19)


## Embedding comparison features
For each pair: |JD − resume| and JD × resume from the saved Sentence-BERT embeddings (768 values). The raw resume vector is not used, so the model cannot memorise resumes.

In [3]:
E = joblib.load(f"{data_dir}/embeddings.joblib")
emb = dict(zip(E["texts"], E["embeddings"]))

def pair_feats(df):
    J = np.stack(df["jd"].map(emb).values)
    R = np.stack(df["resume"].map(emb).values)
    return np.hstack([np.abs(J - R), J * R])

Xe_tr, Xe_va = pair_feats(train), pair_feats(val)
print("Embedding comparison features:", Xe_tr.shape, Xe_va.shape)

Embedding comparison features: (5285, 768) (943, 768)


## Model 1: Sentence-BERT model (Logistic Regression on embedding comparisons)

In [4]:
sb_runs = []
for C in [0.01, 0.1, 1]:
    m = make_pipeline(StandardScaler(), LogisticRegression(C=C, class_weight="balanced", max_iter=3000))
    t = time.time(); m.fit(Xe_tr, y_tr); fit_s = time.time() - t
    f1 = report(f"SBERT LogReg C={C}", m.predict(Xe_va), "embedding comparisons (768)", f"C={C}", fit_s, verbose=False)
    sb_runs.append((f1, m, C))

best_sb_f1, best_sb, best_sb_C = max(sb_runs, key=lambda r: r[0])
print(f"\nBest SBERT model: C={best_sb_C}, macro F1 {best_sb_f1:.3f}")
print(classification_report(y_va, best_sb.predict(Xe_va), labels=order, digits=3))
joblib.dump(best_sb, f"{model_dir}/sbert_logreg.joblib")

SBERT LogReg C=0.01: macro F1 0.527 | accuracy 0.558
SBERT LogReg C=0.1: macro F1 0.523 | accuracy 0.554
SBERT LogReg C=1: macro F1 0.510 | accuracy 0.540

Best SBERT model: C=0.01, macro F1 0.527
               precision    recall  f1-score   support

       No Fit      0.728     0.655     0.690       458
Potential Fit      0.373     0.442     0.405       233
     Good Fit      0.482     0.488     0.485       252

     accuracy                          0.558       943
    macro avg      0.528     0.528     0.527       943
 weighted avg      0.575     0.558     0.565       943



['/content/drive/MyDrive/resume-jd-matching/models/sbert_logreg.joblib']

## Model 2: XGBoost hybrid
14 engineered features + embedding comparisons compressed to 32 values (PCA, fitted on train only).
Also run with the 14 features alone, to measure what the embedding part adds.

In [5]:
pca = PCA(n_components=32, random_state=42).fit(Xe_tr)
P_tr, P_va = pca.transform(Xe_tr), pca.transform(Xe_va)
print("PCA variance explained:", round(pca.explained_variance_ratio_.sum(), 3))

lab_map = {l: i for i, l in enumerate(order)}
yn_tr, yn_va = y_tr.map(lab_map).values, y_va.map(lab_map).values
sw = compute_sample_weight("balanced", yn_tr)

sets = {
    "14 features": (train[FEATS].values, val[FEATS].values),
    "14 features + 32 PCA": (np.hstack([train[FEATS].values, P_tr]), np.hstack([val[FEATS].values, P_va])),
}

xgb_runs = []
for set_name, (Xtr, Xva) in sets.items():
    for depth in [3, 5]:
        m = XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=depth, subsample=0.8,
                          colsample_bytree=0.8, objective="multi:softprob", eval_metric="mlogloss",
                          early_stopping_rounds=50, random_state=42, n_jobs=-1)
        t = time.time()
        m.fit(Xtr, yn_tr, sample_weight=sw, eval_set=[(Xva, yn_va)], verbose=False)
        fit_s = time.time() - t
        pred = np.array(order)[m.predict(Xva)]
        p = f"depth={depth}, lr=0.05, trees={m.best_iteration + 1}"
        f1 = report(f"XGBoost {set_name} depth={depth}", pred, set_name, p, fit_s, verbose=False)
        xgb_runs.append((f1, m, set_name, depth))

best_xgb_f1, best_xgb, best_set, best_depth = max(xgb_runs, key=lambda r: r[0])
print(f"\nBest XGBoost: {best_set}, depth={best_depth}, macro F1 {best_xgb_f1:.3f}")
Xva_best = sets[best_set][1]
print(classification_report(y_va, np.array(order)[best_xgb.predict(Xva_best)], labels=order, digits=3))

names = FEATS + ([f"emb_pca_{i}" for i in range(32)] if "PCA" in best_set else [])
gain = pd.Series(best_xgb.get_booster().get_score(importance_type="gain"))
gain.index = [names[int(k[1:])] for k in gain.index]
print("\nTop 15 features by gain:\n", gain.sort_values(ascending=False).head(15).round(2).to_string())

PCA variance explained: 0.301
XGBoost 14 features depth=3: macro F1 0.486 | accuracy 0.498
XGBoost 14 features depth=5: macro F1 0.497 | accuracy 0.511
XGBoost 14 features + 32 PCA depth=3: macro F1 0.528 | accuracy 0.555
XGBoost 14 features + 32 PCA depth=5: macro F1 0.545 | accuracy 0.583

Best XGBoost: 14 features + 32 PCA, depth=5, macro F1 0.545
               precision    recall  f1-score   support

       No Fit      0.706     0.718     0.712       458
Potential Fit      0.386     0.536     0.449       233
     Good Fit      0.627     0.381     0.474       252

     accuracy                          0.583       943
    macro avg      0.573     0.545     0.545       943
 weighted avg      0.606     0.583     0.583       943


Top 15 features by gain:
 jd_years            6.53
jd_years_missing    5.24
jd_skill_count      4.98
exp_gap             4.92
tfidf_cosine        4.27
emb_pca_29          4.01
emb_pca_0           4.00
exp_meets           3.99
emb_pca_10          3.90
missing

## Save and compare

In [6]:
best_xgb.save_model(f"{model_dir}/xgb_hybrid.json")   # JSON format loads across XGBoost versions
joblib.dump({"pca": pca, "feature_set": best_set, "features": FEATS, "order": order},
            f"{model_dir}/xgb_hybrid_meta.joblib")

log = pd.read_csv(log_path)
print(log.sort_values("val_macro_f1", ascending=False)
      [["notebook", "model", "val_macro_f1", "val_accuracy"]].head(12).to_string(index=False))

 notebook                                model  val_macro_f1  val_accuracy
        5 XGBoost 14 features + 32 PCA depth=5        0.5450        0.5832
        5 XGBoost 14 features + 32 PCA depth=3        0.5285        0.5546
        5                  SBERT LogReg C=0.01        0.5265        0.5578
        5                   SBERT LogReg C=0.1        0.5233        0.5536
        5                     SBERT LogReg C=1        0.5102        0.5398
        4      RandomForest depth=None, leaf=5        0.4980        0.5196
        5          XGBoost 14 features depth=5        0.4966        0.5111
        4        RandomForest depth=10, leaf=1        0.4898        0.5111
        4        RandomForest depth=10, leaf=5        0.4863        0.5005
        5          XGBoost 14 features depth=3        0.4857        0.4984
        4      RandomForest depth=None, leaf=1        0.4848        0.5493
        4       RandomForest depth=10, leaf=20        0.4831        0.4952


## Advanced model findings (validation)
- Sentence-BERT model (LogReg on |JD−resume| and JD×resume): macro F1 0.527 — beats all baselines and the resume-only check (0.426) by 0.10.
- XGBoost hybrid (14 features + 32 PCA embedding components, depth 5): macro F1 0.545, accuracy 0.583 — best model.
- Ablation: XGBoost on 14 features alone 0.497 → +0.048 from embedding components.
- Good Fit: precision 0.63, recall 0.38 — the model is cautious about predicting Good Fit.
- Experience features rank high by gain despite weak correlation → they help in combination with other features.
- PCA keeps 30% of variance; more components is possible future work.

## Model 3: DistilBERT cross-encoder
JD and resume read together as one input, truncated to 512 tokens (longest side trimmed first).
Fine-tuned 2 epochs, learning rate 2e-5, batch 16, class-weighted loss. Best epoch chosen on validation.

In [2]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

MODEL_NAME = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
lab_map = {l: i for i, l in enumerate(order)}

def encode(df):
    enc = tok(list(df["jd"]), list(df["resume"]), truncation="longest_first",
              max_length=512, padding="max_length", return_tensors="pt")
    y = torch.tensor(df["label"].map(lab_map).values)
    return TensorDataset(enc["input_ids"], enc["attention_mask"], y)

ds_tr, ds_va = encode(train), encode(val)
dl_tr = DataLoader(ds_tr, batch_size=16, shuffle=True)
dl_va = DataLoader(ds_va, batch_size=32)
print("Train batches:", len(dl_tr), "| val batches:", len(dl_va))

Device: cuda


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Train batches: 331 | val batches: 30


In [5]:
torch.manual_seed(42)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(device)

counts = train["label"].map(lab_map).value_counts().sort_index().values
class_w = torch.tensor(len(train) / (3 * counts), dtype=torch.float).to(device)
loss_fn = nn.CrossEntropyLoss(weight=class_w)

EPOCHS = 4
opt = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = EPOCHS * len(dl_tr)
sched = get_linear_schedule_with_warmup(opt, int(0.1 * total_steps), total_steps)
use_amp = device == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

def predict(dl):
    model.eval(); out = []
    with torch.no_grad():
        for ids, mask, _ in dl:
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
            out.append(logits.float().softmax(-1).cpu())
    return torch.cat(out).numpy()

best = (-1, None, None, None)
t0 = time.time()
for ep in range(EPOCHS):
    model.train(); total = 0.0
    for ids, mask, y in dl_tr:
        opt.zero_grad()
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
        loss = loss_fn(logits.float(), y.to(device))
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        total += loss.item()
    probs = predict(dl_va)
    f1 = f1_score(y_va, np.array(order)[probs.argmax(1)], average="macro")
    print(f"Epoch {ep + 1}: train loss {total / len(dl_tr):.4f} | val macro F1 {f1:.3f} | {time.time() - t0:.0f}s")
    if f1 > best[0]:
        best = (f1, ep + 1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, probs)
fit_s = time.time() - t0

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1: train loss 0.9967 | val macro F1 0.248 | 73s
Epoch 2: train loss 0.8022 | val macro F1 0.291 | 146s
Epoch 3: train loss 0.7548 | val macro F1 0.373 | 218s
Epoch 4: train loss 0.7166 | val macro F1 0.361 | 290s


In [6]:
best_f1, best_ep, state, probs = best
model.load_state_dict(state)
report("DistilBERT cross-encoder", np.array(order)[probs.argmax(1)], "JD+resume text (512 tokens)",
       f"epochs={EPOCHS}, best_epoch={best_ep}, lr=2e-5, batch=16", fit_s)

save_dir = f"{model_dir}/distilbert"
model.save_pretrained(save_dir); tok.save_pretrained(save_dir)
np.save(f"{data_dir}/distilbert_val_probs.npy", probs)

log = pd.read_csv(log_path)
print(log.sort_values("val_macro_f1", ascending=False)
      [["notebook", "model", "val_macro_f1", "val_accuracy"]].head(8).to_string(index=False))

DistilBERT cross-encoder: macro F1 0.373 | accuracy 0.420
               precision    recall  f1-score   support

       No Fit      0.516     0.576     0.544       458
Potential Fit      0.270     0.326     0.296       233
     Good Fit      0.373     0.222     0.279       252

     accuracy                          0.420       943
    macro avg      0.386     0.375     0.373       943
 weighted avg      0.417     0.420     0.412       943



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

 notebook                                model  val_macro_f1  val_accuracy
        5 XGBoost 14 features + 32 PCA depth=5        0.5450        0.5832
        5 XGBoost 14 features + 32 PCA depth=3        0.5285        0.5546
        5                  SBERT LogReg C=0.01        0.5265        0.5578
        5                   SBERT LogReg C=0.1        0.5233        0.5536
        5                     SBERT LogReg C=1        0.5102        0.5398
        4      RandomForest depth=None, leaf=5        0.4980        0.5196
        5          XGBoost 14 features depth=5        0.4966        0.5111
        4        RandomForest depth=10, leaf=1        0.4898        0.5111


## DistilBERT findings (validation)
- 2 epochs: macro F1 0.333 (undertrained). 4 epochs: best 0.373 at epoch 3, then levelled off (epoch 4: 0.361).
- Below the keyword baseline (0.408) and far below the XGBoost hybrid (0.545).
- Reasons: 512-token truncation (sees ~half of each pair), only 240 distinct training JDs, and pretrained sentence embeddings already capture meaning.
- Conclusion: for long texts and limited data, embeddings + engineered features + XGBoost outperform fine-tuning a transformer.